# Round39 9338 Hybrid Repair

Notebook para correr uma ronda focada apenas no `9338.png`.

A estrategia desta ronda e recuperar pose lateral, escamas coloridas e barriga teal sem voltar a usar termos como `dragon` ou `wing`.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "search_multiseed_validate.py").exists():
    PROJECT_ROOT = Path("C:\\Users\\tugap\\Desktop\\Universidade\\Masters2\u00baAno\\IAG\\ProjetoCunha\\Projeto2")

SRC_DIR = PROJECT_ROOT / "src"
PROMPT_BANK = PROJECT_ROOT / "prompts" / "refinement_round39_9338_hybrid.json"
TARGETS_DIR = PROJECT_ROOT / "TP2-students" / "students" / "tp2-chosen"
OUTPUT_DIR = PROJECT_ROOT / "TP2-students" / "students" / "outputs"

PYTHON_CANDIDATES = [
    PROJECT_ROOT / ".venv_win" / "Scripts" / "python.exe",
    Path("C:\\Users\\tugap\\Desktop\\Universidade\\Masters2\u00baAno\\IAG\\Projeto 2\\.venv\\Scripts\\python.exe"),
    Path(sys.executable),
]

def has_module(python_exe, module_name):
    if not Path(python_exe).exists():
        return False
    result = subprocess.run(
        [str(python_exe), "-c", f"import {module_name}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    return result.returncode == 0

PYTHON_EXE = None
for candidate in PYTHON_CANDIDATES:
    if has_module(candidate, "diffusers"):
        PYTHON_EXE = candidate
        break

if PYTHON_EXE is None:
    raise RuntimeError("No Python with diffusers found. Create .venv_win and install requirements.txt")

print("Project root:", PROJECT_ROOT)
print("Notebook kernel:", sys.executable)
print("Render/search Python:", PYTHON_EXE)
print("Prompt bank:", PROMPT_BANK)
assert (SRC_DIR / "generate_round39_9338_hybrid.py").exists()
assert (SRC_DIR / "search_multiseed_validate.py").exists()
assert TARGETS_DIR.exists()

Project root: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2
Notebook kernel: c:\Users\tugap\AppData\Local\Programs\Python\Python311\python.exe
Render/search Python: C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe
Prompt bank: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\refinement_round39_9338_hybrid.json


## Configuracao

In [2]:
RUN_MODE = "focused"  # smoke | focused | deep

MODES = {
    "smoke": {
        "identity": "round39_9338_smoke",
        "limit_per_target": 24,
        "top_k": 4,
        "stage1_save_k": 8,
        "validation_top_n": 6,
        "ensemble_per_metric": 2,
        "seed_offsets": [1],
    },
    "focused": {
        "identity": "round39_9338_hybrid",
        "limit_per_target": None,
        "top_k": 12,
        "stage1_save_k": 36,
        "validation_top_n": 24,
        "ensemble_per_metric": 6,
        "seed_offsets": [1, 2, 3],
    },
    "deep": {
        "identity": "round39_9338_hybrid_deep",
        "limit_per_target": None,
        "top_k": 16,
        "stage1_save_k": 48,
        "validation_top_n": 36,
        "ensemble_per_metric": 8,
        "seed_offsets": [1, 2, 3, 4, 5],
    },
}

config = MODES[RUN_MODE]
print("Selected mode:", RUN_MODE)
print(json.dumps(config, indent=2))

Selected mode: focused
{
  "identity": "round39_9338_hybrid",
  "limit_per_target": null,
  "top_k": 12,
  "stage1_save_k": 36,
  "validation_top_n": 24,
  "ensemble_per_metric": 6,
  "seed_offsets": [
    1,
    2,
    3
  ]
}


## Gerar prompts

In [3]:
cmd = [str(PYTHON_EXE), str(SRC_DIR / "generate_round39_9338_hybrid.py"), "--output", str(PROMPT_BANK)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

data = json.loads(PROMPT_BANK.read_text(encoding="utf-8"))
print({target: len(entries) for target, entries in data.items()})

Running: C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\src\generate_round39_9338_hybrid.py --output c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\refinement_round39_9338_hybrid.json
{'9338.png': 720}


## Correr pesquisa

In [4]:
args = [
    str(PYTHON_EXE),
    str(SRC_DIR / "search_multiseed_validate.py"),
    "--prompts", str(PROMPT_BANK),
    "--targets", str(TARGETS_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--identity", config["identity"],
    "--only", "9338.png",
    "--top-k", str(config["top_k"]),
    "--stage1-save-k", str(config["stage1_save_k"]),
    "--validation-top-n", str(config["validation_top_n"]),
    "--ensemble-per-metric", str(config["ensemble_per_metric"]),
    "--seed-offsets", *[str(seed) for seed in config["seed_offsets"]],
    "--offline",
    "--disable-progress-bar",
    "--maxstack-scoring",
]
if config["limit_per_target"] is not None:
    args.extend(["--limit-per-target", str(config["limit_per_target"])])

env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"

print("Running:")
print(" ".join(args))
process = subprocess.Popen(
    args,
    cwd=PROJECT_ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Search failed with exit code {return_code}")
print("Finished successfully")

Running:
C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\src\search_multiseed_validate.py --prompts c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\refinement_round39_9338_hybrid.json --targets c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\tp2-chosen --output-dir c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\outputs --identity round39_9338_hybrid --only 9338.png --top-k 12 --stage1-save-k 36 --validation-top-n 24 --ensemble-per-metric 6 --seed-offsets 1 2 3 --offline --disable-progress-bar --maxstack-scoring
Couldn't connect to the Hub: Cannot reach https://huggingface.co/api/models/SimianLuo/LCM_Dreamshaper_v7: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable..
Will try to load from local cac

## Ver resultados

In [ ]:
run_dirs = sorted(
    [path for path in OUTPUT_DIR.glob(f"*_{config['identity']}") if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not run_dirs:
    print("No run directory found")
else:
    latest = run_dirs[0]
    print("Latest run:", latest)
    for item in sorted(latest.glob("*")):
        print(item.name)

Latest run: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\outputs\20260602-230347_round39_9338_hybrid
9338
9338_robust_prompt_ranking.csv
9338_selected_for_multiseed.csv
9338_stage1_fixed_seed_top36.csv
contact_sheet_top12_robust.jpg
stage1_fixed_seed_metrics.csv
stage2_aux_seed_metrics.csv
summary.json
top12_robust_fixed_seed.csv


: 